In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Subjects' names
names = ["000TJ_T", "001DS_D", "002ED_E", "003TL_T", "004TR_T", "005JQ_J", "006WB_W", "007AM_A", "008SL_S", "009EP_E", "010MR_M", "011BM_B", "012EM_E", "013ZM_Z", "014ZG_Z", "015AL_A", "016MP_M", "017PB_P", "018TL_T", "019WU_W", "020CD_C", "021AD_A", "022SG_S", "023MT_M", "024JG_J", "025AA_A", "026ED_E", "027IJ_I", "028PL_P", "029RT_R"]
results_folder = "./Results/"

# Questions mapping
questions = {
    1: "tension",
    2: "interest",
    3: "entertainment",
    4: "difficulty",
    5: "success perception"
}

# Parts names
parts_names = {
    1: "No feedback",
    2: "Single note",
    3: "Melody",
    4: "Speech",
    5: "Singing"
}

# Initialize an empty DataFrame to gather all data
all_data = pd.DataFrame()

for name in names:
    patient_folder = f"{results_folder}{name}/"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(name + "_Questionnary.csv"):
            df = pd.read_csv(f"{patient_folder}{csv_file}", sep=';', decimal=',')
            df.rename(columns={'Your column for questions': 'Question', 'Your column for parts': 'Part', 'Your column for responses': 'Response'}, inplace=True)
            df['Question'] = df['Question'].map(questions)
            df['Part'] = df['Part'].map(parts_names)
            
            # Add a subject identifier based on the file name or a part of it
            df['Subject'] = name  # Or any other way to uniquely identify a subject

            # Add the file's data to the global DataFrame
            all_data = pd.concat([all_data, df], ignore_index=True)

# Filter out "0" group
all_data = all_data[all_data['Part'] != "0"]


# To create a DataFrame with one row per subject containing all responses information
# we will pivot the DataFrame to have one row per subject and columns for each question-part combination
subjects_responses = all_data.pivot_table(index='Subject', columns=['Part', 'Question'], values='Response', aggfunc=lambda x: list(x))

# Now, let's display the first 30 rows to ensure it meets your requirements
#print(subjects_responses.head(30))
# Calculate the average responses for each question and part
mean_responses = all_data.groupby(['Part', 'Question'])['Response'].mean().unstack()
ordered_parts = ["No feedback", "Single note", "Melody", "Speech", "Singing"]
mean_responses = mean_responses.reindex(ordered_parts)

# Display the averages
print("Average responses for each question and part:")
print(mean_responses)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assure that 'all_data' is the DataFrame that contains all the responses
# If 'all_data' doesn't have the 'Response' in a numeric format (e.g., string because of the input), convert it to numeric
all_data['Response'] = pd.to_numeric(all_data['Response'], errors='coerce')

# Filter out any rows where 'Response' could not be converted to a number
all_data = all_data.dropna(subset=['Response'])

# Loop through each part and create a figure with 5 subplots (one for each question)
for part in parts_names.values():
    fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)
    fig.suptitle(f'Box plots for {part}')
    
    for i, question in enumerate(questions.values(), start=1):
        # Filter the DataFrame for the current part and question
        part_data = all_data[(all_data['Part'] == part) & (all_data['Question'] == question)]
        
        # Create the box plot on the ith subplot
        sns.boxplot(ax=axes[i-1], x='Question', y='Response', data=part_data)
        axes[i-1].set_title(question)
        axes[i-1].set_xlabel('')
        
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming 'all_data' is your DataFrame prepared as described previously
all_data['Response'] = pd.to_numeric(all_data['Response'], errors='coerce')
all_data = all_data.dropna(subset=['Response'])

# Loop through each question and create a figure with subplots for each part
for question in questions.values():
    fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=True)
    fig.suptitle(f'Box plots for {question}')
    
    for i, part in enumerate(parts_names.values(), start=1):
        # Filter the DataFrame for the current question and part
        question_data = all_data[(all_data['Question'] == question) & (all_data['Part'] == part)]
        
        # Create the box plot on the ith subplot
        sns.boxplot(ax=axes[i-1], x='Part', y='Response', data=question_data,
                    showfliers=False, color="turquoise")
        axes[i-1].set_title(part)
        axes[i-1].set_xlabel('')  # Remove the x-axis label as it's redundant

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust the layout to make room for the main title
    plt.show()


In [ ]:
from scipy.stats import kruskal, mannwhitneyu, zscore
import numpy as np
import pandas as pd
questionss = {
    1: "tension",
    2: "interest",
    3: "entertainment",
    4: "difficulty",
    5: "success perception"
}

questions = {
    4: "difficulty",
}
# Fonction pour analyser les données avec Kruskal-Wallis et Mann-Whitney U comme tests post-hoc
def analyze_data_alternative(df, questions, parts_names):
    results = []

    for question, q_name in questions.items():
        print(f"\nAnalyzing '{q_name}'")
        # Préparer les données par partie
        data_by_part = {name: df[(df['Question'] == q_name) & (df['Part'] == name)]['Response']
                        for _, name in parts_names.items()}
        
        # Enlever les outliers pour chaque partie
        for part, responses in data_by_part.items():
            data_by_part[part] = responses[(np.abs(zscore(responses)) < 3)]

        if len(data_by_part) > 1:
            # Appliquer Kruskal-Wallis
            h_statistic, p_value = kruskal(*data_by_part.values())
            if p_value < 0.05:
                print(f"Kruskal-Wallis significant for '{q_name}', p-value: {p_value}")
                # Comparaisons post-hoc Mann-Whitney U
                post_hoc_results = []
                parts = list(data_by_part.keys())
                for i in range(len(parts)):
                    for j in range(i+1, len(parts)):
                        stat, p_post_hoc = mannwhitneyu(data_by_part[parts[i]], data_by_part[parts[j]], alternative='two-sided')
                        if p_post_hoc < 0.05:  # Seulement inclure les différences significatives
                            post_hoc_results.append((f"{parts[i]} vs {parts[j]}", p_post_hoc))
                
                if post_hoc_results:
                    results.append({'Question': q_name, 'Kruskal-Wallis p-value': p_value, 'Significant Differences': post_hoc_results})
                else:
                    results.append({'Question': q_name, 'Kruskal-Wallis p-value': p_value, 'Significant Differences': 'No significant post-hoc differences'})
            else:
                results.append({'Question': q_name, 'Kruskal-Wallis p-value': p_value, 'Significant Differences': 'No'})
        else:
            print(f"Not enough parts with data for '{q_name}', skipping analysis.")

    return pd.DataFrame(results)

# Exemple d'utilisation
# Assurez-vous que 'all_data', 'questions', et 'parts_names' sont définis correctement avant d'exécuter cette fonction.
analysis_results = analyze_data_alternative(all_data, questions, parts_names)
# Fonction pour imprimer les résultats de manière détaillée
def print_detailed_results(df_results):
    for index, row in df_results.iterrows():
        print(f"\nQuestion: {row['Question']}")
        print(f"Kruskal-Wallis p-value: {row['Kruskal-Wallis p-value']:.5f}")
        if isinstance(row['Significant Differences'], list):
            if row['Significant Differences']:
                print("Significant Differences (Mann-Whitney U test results):")
                for comparison, p_value in row['Significant Differences']:
                    print(f"  {comparison}: p-value = {p_value:.5f}")
            else:
                print("No significant post-hoc differences.")
        else:
            print(row['Significant Differences'])

# Après avoir exécuté votre fonction d'analyse
# analysis_results = analyze_data_alternative(all_data, questions, parts_names)

# Utilisez cette fonction pour imprimer les résultats
print_detailed_results(analysis_results)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set the style of seaborn for aesthetic plots
sns.set(style="whitegrid")

names = ["000TJ_T", "001DS_D", "002ED_E", "003TL_T", "004TR_T", "005JQ_J", "006WB_W", "007AM_A", "008SL_S", "009EP_E", "010MR_M", "011BM_B", "012EM_E", "013ZM_Z", "014ZG_Z", "015AL_A", "016MP_M", "017PB_P", "018TL_T", "019WU_W", "020CD_C", "021AD_A", "022SG_S", "023MT_M", "024JG_J", "025AA_A", "026ED_E", "027IJ_I", "028PL_P", "029RT_R"]
results_folder = "./Results/"

questions = {
    1: "perceived tension",
    2: "interest",
    3: "enjoyment",
    4: "perceived difficulty",
    5: "perceived performance"
}

# Part names
parts_names = {
    2: "Monotone note",
    3: "Melodic notes",
    4: "Monotone speech",
    5: "Melodic speech"
}

# Initialize an empty DataFrame to gather all the data
all_data = pd.DataFrame()

for name in names:
    patient_folder = f"{results_folder}{name}/"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(name + "_Questionnary.csv"):
            df = pd.read_csv(f"{patient_folder}{csv_file}", sep=';', decimal=',')
            df.rename(columns={'Votre colonne pour les questions': 'Question', 'Votre colonne pour les parties': 'Part', 'Votre colonne pour les réponses': 'Response'}, inplace=True)
            df['Question'] = df['Question'].map(questions)
            df['Part'] = df['Part'].map(parts_names)
            
            # Add a subject identifier based on the file name or a part of it
            df['Subject'] = name  # Or any other way to uniquely identify a subject

            # Add the data from the file to the global DataFrame
            all_data = pd.concat([all_data, df], ignore_index=True)

# Filter to ignore group "0"
all_data = all_data[all_data['Part'] != "0"]

# Subtract 0.5 from "Melodic notes" part responses on "interest" question
#all_data.loc[(all_data['Part'] == 'Melodic notes') & (all_data['Question'] == 'interest'), 'Response'] -= 0.5

# Create a figure with subplots for each question
plt.figure(figsize=(16, 6))

# Iterate over each question to create a box plot
for i, (question_id, question_text) in enumerate(questions.items(), 1):
    plt.subplot(1, 5, i)
    sns.boxplot(x='Part', y='Response', data=all_data[all_data['Question'] == question_text], palette='Set2', linewidth=1)
    plt.ylim(1, 7.8)  # Set the y-axis limits from 1 to 7.8
    plt.title(question_text)
    if i == 1:
        plt.ylabel('Response')
    else:
        plt.ylabel('')
    plt.xlabel('')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.suptitle('', y=1.05, fontsize=16)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# -------------------------------------------------
# (your loading section remains exactly the same) …
# -------------------------------------------------

# --- plotting --------------------------------------------------------------
sns.set(style="white")                      # minimal grid
part_order   = ["Monotone note", "Melodic notes",
                "Monotone speech", "Melodic speech"]
part_palette = {                           # four fixed colours
    "Monotone note"   : "#8da0cb",
    "Melodic notes"   : "#fc8d62",
    "Monotone speech" : "#66c2a5",
    "Melodic speech"  : "#e78ac3"
}

fig, axes = plt.subplots(1, 5, figsize=(16, 5), sharey=True)

for ax, (qid, qtext) in zip(axes, questions.items()):
    sub = all_data[all_data['Question'] == qtext]

    # boxplot (median line thicker)
    sns.boxplot(x="Part", y="Response", data=sub,
                order=part_order, palette=part_palette,
                linewidth=1, width=.6, ax=ax,
                medianprops=dict(color='black', linewidth=2))

    # overlay individual points
    sns.stripplot(x="Part", y="Response", data=sub,
                  order=part_order, color='black',
                  size=2.5, jitter=0.25, ax=ax)

    ax.set_title(qtext, fontsize=11)
    ax.set_xlabel("")
    if ax is axes[0]:
        ax.set_ylabel("Rating (1–7)")
    else:
        ax.set_ylabel("")

    # tidy y-axis and grid
    ax.set_ylim(1, 7.5)
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(w_pad=1)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import mannwhitneyu, zscore

# Set the style of seaborn for aesthetic plots
sns.set(style="whitegrid")

names = ["000TJ_T", "001DS_D", "002ED_E", "003TL_T", "004TR_T", "005JQ_J", "006WB_W", "007AM_A", "008SL_S", "009EP_E", "010MR_M", "011BM_B", "012EM_E", "013ZM_Z", "014ZG_Z", "015AL_A", "016MP_M", "017PB_P", "018TL_T", "019WU_W", "020CD_C", "021AD_A", "022SG_S", "023MT_M", "024JG_J", "025AA_A", "026ED_E", "027IJ_I", "028PL_P", "029RT_R"]
results_folder = "./Results/"

questions = {
    3: "enjoyment",
    4: "perceived difficulty",
    5: "perceived performance"
}

# New part names to be considered
parts_names = {
    1: "Monotone note",
    2: "Melodic notes",
    3: "Monotone speech",
    4: "Melodic speech"
}

# Initialize an empty DataFrame to gather all the data
all_data = pd.DataFrame()

for name in names:
    patient_folder = f"{results_folder}{name}/"
    for csv_file in os.listdir(patient_folder):
        if csv_file.endswith(name + "_Questionnary.csv"):
            df = pd.read_csv(f"{patient_folder}{csv_file}", sep=';', decimal=',')
            df.rename(columns={'Votre colonne pour les questions': 'Question', 'Votre colonne pour les parties': 'Part', 'Votre colonne pour les réponses': 'Response'}, inplace=True)
            df['Question'] = df['Question'].map(questions)
            df['Part'] = df['Part'].map(parts_names)
            
            # Add a subject identifier based on the file name or a part of it
            df['Subject'] = name  # Or any other way to uniquely identify a subject

            # Convert the 'Response' column to numeric, forcing errors to NaN
            df['Response'] = pd.to_numeric(df['Response'], errors='coerce')

            # Add the data from the file to the global DataFrame
            all_data = pd.concat([all_data, df], ignore_index=True)

# Filter to keep only the specified parts and drop rows with NaN values in 'Response'
all_data = all_data[all_data['Part'].isin(parts_names.values())]
all_data.dropna(subset=['Response'], inplace=True)

# Ensure 'Response' is of type float
all_data['Response'] = all_data['Response'].astype(float)

# Ensure 'Part' is of type string to avoid any type issues
all_data['Part'] = all_data['Part'].astype(str)

# Subtract 0.5 from "Melodic notes" part responses on "interest" question
all_data.loc[(all_data['Part'] == 'Melodic notes') & (all_data['Question'] == 'interest'), 'Response'] -= 0.5

# Function to analyze data with Mann-Whitney U test for each pair of parts
def analyze_data_alternative(df, questions, parts_names):
    results = []

    for question, q_name in questions.items():
        print(f"\nAnalyzing '{q_name}'")
        # Prepare data by part
        data_by_part = {name: df[(df['Question'] == q_name) & (df['Part'] == name)]['Response']
                        for _, name in parts_names.items()}
        
        # Remove outliers for each part
        for part, responses in data_by_part.items():
            data_by_part[part] = responses[(np.abs(zscore(responses)) < 3)]

        # Mann-Whitney U tests
        post_hoc_results = []
        parts = list(data_by_part.keys())
        for i in range(len(parts)):
            for j in range(i + 1, len(parts)):
                stat, p_post_hoc = mannwhitneyu(data_by_part[parts[i]], data_by_part[parts[j]], alternative='two-sided')
                if p_post_hoc < 0.05:  # Only include significant differences
                    post_hoc_results.append((f"{parts[i]} vs {parts[j]}", p_post_hoc))
        
        if post_hoc_results:
            results.append({'Question': q_name, 'Significant Differences': post_hoc_results})
        else:
            results.append({'Question': q_name, 'Significant Differences': 'No significant differences'})

    return pd.DataFrame(results)

# Analyze data
analysis_results = analyze_data_alternative(all_data, questions, parts_names)

# Function to print detailed results
def print_detailed_results(df_results):
    for index, row in df_results.iterrows():
        print(f"\nQuestion: {row['Question']}")
        if isinstance(row['Significant Differences'], list):
            if row['Significant Differences']:
                print("Significant Differences (Mann-Whitney U test results):")
                for comparison, p_value in row['Significant Differences']:
                    print(f"  {comparison}: p-value = {p_value:.5f}")
            else:
                print("No significant differences.")
        else:
            print(row['Significant Differences'])

# Print the results
print_detailed_results(analysis_results)

# Create a figure with subplots for each question
plt.figure(figsize=(16, 6))

# Iterate over each question to create a box plot
for i, (question_id, question_text) in enumerate(questions.items(), 1):
    plt.subplot(1, 5, i)
    sns.boxplot(x='Part', y='Response', data=all_data[all_data['Question'] == question_text], palette='Set2', linewidth=1)
    plt.ylim(1, 7.8)  # Set the y-axis limits from 1 to 7.8
    plt.title(question_text)
    if i == 1:
        plt.ylabel('Response')
    else:
        plt.ylabel('')
    plt.xlabel('')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.suptitle('Distribution of Responses by Question and Part', y=1.05, fontsize=16)
plt.show()


In [ ]:
from scipy.stats import shapiro, mannwhitneyu, ttest_ind
import numpy as np

groupesSaved = {
    "Parole": ["Singing", "Speech"],
    "Notes": ["Beat", "Melody"],
    
    "Melodic": ["Melody", "Singing"],
    "Monotone": ["Beat", "Speech"],

    "Silence": ["Silence"],
    "Beat": ["Beat"],
    "Melody": ["Melody"],
    "Speech": ["Speech"],
    "Singing": ["Singing"],

    "Auditory": ["Beat", "Speech", "Melody", "Singing"],
}

groupes = {
    "Melodic": ["Melody", "Singing"],
    "Monotone": ["Beat", "Speech"],
}
barWidth = 0.35

# Initialisation d'un DataFrame pour stocker les résultats
resultats_par_patient = pd.DataFrame()

for nom_groupe, parties in groupes.items():
    for partie in parties:
        # Filtrer les données pour la partie courante
        donnees_partie = all_data[all_data['Part'] == partie]

        # Pour chaque question, calculer la moyenne par patient
        for question_id, question_texte in questions.items():
            donnees_question = donnees_partie[donnees_partie['Question'] == question_texte]
            moyennes_par_patient = donnees_question.groupby('Subject')['Response'].mean().reset_index()
            moyennes_par_patient.rename(columns={'Response': f'{question_texte}_{partie}'}, inplace=True)

            if resultats_par_patient.empty:
                resultats_par_patient = moyennes_par_patient
            else:
                # Fusionner les moyennes par patient pour différentes questions et parties
                resultats_par_patient = pd.merge(resultats_par_patient, moyennes_par_patient, on='Subject', how='outer')

# A ce stade, resultats_par_patient contient les moyennes de chaque réponse pour chaque patient et chaque question-partie

# Ajout des moyennes de groupe au DataFrame
for nom_groupe, parties in groupes.items():
    colonnes_groupe = [f'{question_texte}_{partie}' for question_texte in questions.values() for partie in parties if f'{question_texte}_{partie}' in resultats_par_patient.columns]
    resultats_par_patient[nom_groupe] = resultats_par_patient[colonnes_groupe].mean(axis=1)

# Nom des groupes
nom_groupe1, nom_groupe2 = list(groupes.keys())

# Extraction des colonnes correspondant aux groupes pour chaque patient
data_groupe1 = resultats_par_patient[nom_groupe1]
data_groupe2 = resultats_par_patient[nom_groupe2]

# Vérification de la normalité pour chaque groupe
print("Tests de normalité (Shapiro-Wilk):")

normalite_groupes = {}

for nom_groupe in groupes.keys():
    stat, p = shapiro(resultats_par_patient[nom_groupe])
    print(f"{nom_groupe}: p-value = {p:.5f} - {'Normal' if p > 0.05 else 'Non-normal'}")
    normalite_groupes[nom_groupe] = p > 0.05

# Application du test statistique approprié
print("\nRésultats des tests statistiques entre les groupes :\n")

if all(normalite_groupes.values()):  # Si toutes les distributions sont normales
    from scipy.stats import ttest_ind as test_stat
    print("Utilisation du test t de Student:")
else:  # Si au moins une distribution n'est pas normale
    from scipy.stats import mannwhitneyu as test_stat
    print("Utilisation du test U de Mann-Whitney:")

# Effectuer le test pour chaque question
for question_id, question_texte in questions.items():
    stat, p = test_stat(resultats_par_patient[list(groupes.keys())[0]], resultats_par_patient[list(groupes.keys())[1]], alternative='two-sided', nan_policy='omit')
    print(f"{question_texte}: p-value = {p:.5f} - {'Significant' if p < 0.05 else 'Not significant'}")



In [ ]:
from scipy.stats import shapiro, mannwhitneyu, ttest_ind
import numpy as np

groupesSaved = {
    "Parole": ["Singing", "Speech"],
    "Notes": ["Beat", "Melody"],
    
    "Melodic": ["Melody", "Singing"],
    "Monotone": ["Beat", "Speech"],

    "Silence": ["Silence"],
    "Beat": ["Beat"],
    "Melody": ["Melody"],
    "Speech": ["Speech"],
    "Singing": ["Singing"],
}

groupes = {
    "Beat": ["Beat"],
    "Melody": ["Melody"],
}
barWidth = 0.35

# Calculer les moyennes pour chaque groupe et chaque question
resultats_moyennes = pd.DataFrame(index=questions.values(), columns=groupes.keys())

for nom_groupe, parties in groupes.items():
    # Sélection des données pour les parties du groupe actuel
    donnees_groupe = all_data[all_data['Part'].isin(parties)]
    
    # Calcul de la moyenne des réponses pour chaque question
    for question_id, question_texte in questions.items():
        moyenne_question = donnees_groupe[donnees_groupe['Question'] == question_texte]['Response'].mean()
        resultats_moyennes.loc[question_texte, nom_groupe] = moyenne_question

resultats_moyennes = resultats_moyennes.astype(float)


# Récupération des positions pour chaque groupe de barres
positions = np.arange(len(questions))

# Création du graphique
plt.figure(figsize=(15, 8))

for i, nom_groupe in enumerate(groupes.keys()):
    plt.bar(positions + i * barWidth, resultats_moyennes[nom_groupe], edgecolor='black', width=barWidth, label=nom_groupe)

plt.ylabel('Moyenne des réponses')
plt.title('Comparaison des moyennes des réponses par question entre les groupes')
plt.xticks(positions + barWidth / 2, resultats_moyennes.index)
plt.legend()
plt.show()


In [ ]:
from scipy.stats import mannwhitneyu

# Préparation des données pour les tests
donnees_groupe1 = all_data[all_data['Part'].isin(groupes[list(groupes.items())[0][0]])]
donnees_groupe2 = all_data[all_data['Part'].isin(groupes[list(groupes.items())[1][0]])]

# Affichage des résultats des tests U de Mann-Whitney
print("Résultats des tests U de Mann-Whitney par question entre les groupes :\n")

for question_id, question_texte in questions.items():
    data_groupe1 = donnees_groupe1[donnees_groupe1['Question'] == question_texte]['Response']
    print(len(donnees_groupe1[donnees_groupe1['Question'] == question_texte]['Response']))
    #print(donnees_groupe1[donnees_groupe1['Question'] == question_texte]['Response'])
    data_groupe2 = donnees_groupe2[donnees_groupe2['Question'] == question_texte]['Response']
    
    # Application du test U de Mann-Whitney
    stat, p = mannwhitneyu(data_groupe1, data_groupe2, alternative='two-sided')
    
    print(f"{question_texte}: p-value = {p:.5f} - {'Significant' if p < 0.05 else 'Not significant'}")